# 04. AnyIO 与 Trio

`asyncio` 是标准库主流方案；AnyIO 和 Trio 提供了更强调结构化并发的 API。本章目标不是让你背下所有框架，而是理解它们解决的问题和选择边界。

- AnyIO：在 asyncio 或 Trio 后端上提供统一 API；
- Trio：围绕 nursery、取消作用域和结构化并发设计；
- httpx/aiohttp：应用层的异步 HTTP 客户端，并不是新的 Python 调度模型。

## 1. 异步生态分层

```text
应用协议层：HTTPX / aiohttp / 数据库驱动 / Web 框架
并发抽象层：AnyIO
运行时后端：asyncio（标准库）或 Trio
事件循环实现：asyncio 默认循环，可选 uvloop 等实现
```

选择 HTTP 客户端与选择事件循环不是同一件事。例如 HTTPX 的异步接口通常运行在 AnyIO 支持的环境中，而 aiohttp 直接围绕 asyncio 生态构建。

In [1]:
import anyio
import trio


print("AnyIO：", getattr(anyio, "__version__", "版本由包元数据管理"))
print("Trio：", trio.__version__)

AnyIO： 版本由包元数据管理
Trio： 0.32.0


## 2. AnyIO：统一后端 API

AnyIO 提供 task group、取消作用域、同步原语、线程/进程桥接、网络和异步文件等统一接口。应用或库可以使用一套 API，再选择 asyncio 或 Trio 后端。

在 Jupyter 中已有 asyncio 事件循环，因此下面直接使用顶层 `await`，不调用 `anyio.run()`。

In [3]:
async def anyio_worker(name: str, delay: float, results: list[str]) -> None:
    await anyio.sleep(delay)
    results.append(name)


anyio_results: list[str] = []
async with anyio.create_task_group() as task_group:
    task_group.start_soon(anyio_worker, "慢任务", 0.10, anyio_results)
    task_group.start_soon(anyio_worker, "快任务", 0.05, anyio_results)

print(anyio_results)
assert anyio_results == ["快任务", "慢任务"]

['快任务', '慢任务']


## 3. 同一份 AnyIO 代码切换后端

`examples/04_anyio_backends.py` 使用 `anyio.run(..., backend=...)` 启动程序。因为 Jupyter 已经有运行中的事件循环，教程通过子进程分别启动两个后端，这也与命令行应用的真实入口一致。

In [4]:
from pathlib import Path
import subprocess
import sys


anyio_script = Path("examples/04_anyio_backends.py")
for backend in ("asyncio", "trio"):
    completed = subprocess.run(
        [sys.executable, str(anyio_script), "--backend", backend],
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())

当前后端：asyncio
任务 B 完成
任务 A 完成
当前后端：trio
任务 B 完成
任务 A 完成


## 4. Trio：nursery 与结构化并发

Trio 不鼓励任意创建失去父级管理的后台任务。子任务必须在 nursery 中启动：

```python
async with trio.open_nursery() as nursery:
    nursery.start_soon(child, "A")
    nursery.start_soon(child, "B")
```

离开 nursery 前会等待所有子任务；一个子任务出现未处理异常时，nursery 会取消其他子任务并传播错误。任务形成清晰的父子树，因此不会悄悄遗留孤儿任务。

In [5]:
trio_script = Path("examples/05_trio_nursery.py")
completed = subprocess.run(
    [sys.executable, str(trio_script)],
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

失败任务 开始
普通任务 开始
失败任务 清理资源
普通任务 清理资源
nursery 传播的异常： ['失败任务 发生错误']
预期：一个子任务失败后，nursery 会取消仍在运行的兄弟任务。



## 5. 三种 task group 对比

| asyncio | AnyIO | Trio |
|---|---|---|
| `asyncio.TaskGroup()` | `anyio.create_task_group()` | `trio.open_nursery()` |
| `group.create_task(coro())` | `group.start_soon(fn, args...)` | `nursery.start_soon(fn, args...)` |
| 标准库，无额外依赖 | 可切换 asyncio/Trio 后端 | 独立运行时与完整并发模型 |
| Python 3.11+ 提供 TaskGroup | 统一取消、线程、文件和网络 API | nursery/取消作用域是核心设计 |

它们都强调：父级作用域结束时，子任务必须已经完成或被取消。

## 6. httpx 与 aiohttp 放在哪里

项目已经安装二者，但本教程不访问公网：

- `httpx.AsyncClient`：API 与同步 `httpx.Client` 接近，常见于 FastAPI/AnyIO 生态；
- `aiohttp.ClientSession`：asyncio 生态中成熟的 HTTP 客户端，也提供 Web 服务端能力；
- 二者解决 HTTP 通信问题，不取代事件循环、Task 或结构化并发。

真实项目中应复用客户端会话、设置超时、限制连接数，并在 `async with` 中确保关闭资源。

## 7. 其他常见方案的定位

| 方案 | 定位 | 本教程为何不深入 |
|---|---|---|
| uvloop | asyncio 的高性能事件循环实现 | 优化运行时，不改变 async/await 编程模型；有平台边界 |
| Twisted | 历史悠久的事件驱动网络框架 | 体系庞大，现代新项目通常先学习 async/await 主线 |
| gevent | 基于 greenlet 的协作式并发及 monkey patch | 隐式改写阻塞行为，心智模型与原生 async/await 不同 |

维护旧系统时它们都可能重要；新学习路径应先掌握标准库 asyncio，再根据框架需求选择 AnyIO 或 Trio。

## 8. 如何选择

1. 普通 Python 应用且依赖都支持 asyncio：从标准库 asyncio 开始。
2. 编写需要兼容 asyncio/Trio 的库，或使用 FastAPI/Starlette 等 AnyIO 生态：选择 AnyIO。
3. 可以自主决定整个异步栈，并重视严格结构化并发：评估 Trio。
4. 只需要异步 HTTP：先根据现有框架选择 httpx 或 aiohttp，不必更换整个并发模型。
5. 已有同步阻塞库：先考虑线程桥接，而不是强行重写。

## 9. 常见误区与练习

- AnyIO 不是“比 asyncio 更快”的事件循环，它是可运行在不同后端上的抽象层。
- Trio nursery 与 asyncio TaskGroup 概念接近，但取消语义和完整 API 并不完全相同。
- 不能在 Trio 后端中随意调用只支持 asyncio 的库，反之亦然。
- 异步 HTTP 客户端仍需要超时、连接限制、重试策略和资源关闭。

**练习：**运行 `examples/04_anyio_backends.py` 的两个后端，给 task group 新增第三个任务并保持程序逻辑不依赖具体后端。

## 系列小结

- 等待少、流程简单：同步优先。
- 阻塞式 I/O：线程池。
- 纯 Python CPU 计算：进程池。
- 大量原生异步 I/O：asyncio。
- 需要后端抽象和统一结构化并发 API：AnyIO。
- 希望以 nursery 为核心构建完整异步系统：Trio。

异步的价值不是让单个操作更快，而是让程序在等待期间继续推进其他工作，同时保持任务生命周期、错误和取消都可控。